In [ ]:
# ============================================================
# مشروع استخراج وتصحيح نصوص الخط اليدوي — النسخة النهائية
# ============================================================

# الخلية 1: تثبيت المكتبات المطلوبة
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara
!pip install -q pdf2image easyocr pyspellchecker opencv-python-headless transformers torch torchvision

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils tesseract-ocr-ara
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 831 kB of archives.
After this operation, 2,144 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-ara all 1:4.00~git30-7274cfa-1.1 [645 kB]
Fetched 831 kB in 1s (680 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package tesseract-ocr-ara.
Preparing to unpack .../t

In [ ]:
# الخلية 2: استيراد المكتبات
import os
import cv2
import numpy as np
import easyocr
from pdf2image import convert_from_path
from PIL import Image
from spellchecker import SpellChecker
from transformers import pipeline
from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
# الخلية 3: إعداد الأدوات (OCR والمصحح اللغوي)
reader = easyocr.Reader(['ar', 'en'])
spell = SpellChecker(language='ar')
# استخدام نموذج للتحقق من السياق أو التصحيح المتقدم
corrector = pipeline("text2text-generation", model="Zaid/arabic-spell-checker", device=-1)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: Zaid/arabic-spell-checker is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# الخلية 4: تعريف دوال المعالجة
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # تحسين التباين
    enhanced = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    return enhanced

def extract_text(image_path):
    results = reader.readtext(image_path, detail=0)
    return " ".join(results)

In [1]:
# الخلية 5: دالة معالجة PDF
def process_pdf(pages_start=1, pages_end=2, resume=True):
    proc_start = time.time()

    checkpoint = load_checkpoint() if resume else None
    if checkpoint:
        print(f"🔄 استئناف من الصفحة {checkpoint['last_page_processed']}")
        pages_start = checkpoint['last_page_processed']

    correction_dict = build_correction_dict()

    try:
        images = convert_from_path(
            PDF_PATH, dpi=300,
            first_page=pages_start, last_page=pages_end)
    except Exception as e:
        logging.error(f"فشل معالجة PDF: {e}")
        return

    total_words = checkpoint.get('processed_words', 0) if checkpoint else 0

    with sqlite3.connect(DB_PATH) as conn:
        conn.execute('''CREATE TABLE IF NOT EXISTS handwriting_data
            (image_id    INTEGER PRIMARY KEY AUTOINCREMENT,
             image_data  BLOB,
             predicted_text TEXT,
             status      TEXT,
             confidence  REAL,
             model_source TEXT,
             x INTEGER, y INTEGER, w INTEGER, h INTEGER,
             page_num    INTEGER)''')

        # ✅ حذف البيانات القديمة للصفحات المُعاد معالجتها لتجنب التكرار
        conn.execute(
            "DELETE FROM handwriting_data WHERE page_num BETWEEN ? AND ?",
            (pages_start, pages_end))
        conn.commit()
        logging.info(f"🗑️ تم مسح بيانات الصفحات {pages_start}-{pages_end} قبل إعادة المعالجة")

        for idx, pil in enumerate(images):
            actual_page = pages_start + idx
            img_bgr = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

            try:
                detections = easy_reader.readtext(img_bgr, detail=1)
            except:
                detections = []

            binary     = preprocess_image(img_bgr)
            boxes_info = []

            if detections:
                for (bbox, text, conf) in detections:
                    pts = np.array(bbox, dtype=np.int32)
                    x, y, w, h = cv2.boundingRect(pts)
                    boxes_info.append(((x, y, w, h), (bbox, text, conf)))
            else:
                manual_boxes = smart_word_segmentation(img_bgr, binary)
                for box in manual_boxes:
                    boxes_info.append((box, None))

            for (box, e_raw) in boxes_info:
                x, y, w, h = box
                crop = img_bgr[y:y+h, x:x+w]
                raw, conf, src, _ = recognize_word_ensemble(crop, easyocr_raw=e_raw)
                final = apply_correction_dict(correct_text(raw), correction_dict)
                _, buf = cv2.imencode(".png", crop)
                conn.execute(
                    '''INSERT INTO handwriting_data
                       (image_data, predicted_text, status, confidence,
                        model_source, x, y, w, h, page_num)
                       VALUES (?,?,?,?,?,?,?,?,?,?)''',
                    (buf.tobytes(), final, 'unverified', conf, src,
                     x, y, w, h, actual_page))
                total_words += 1

            save_checkpoint(actual_page + 1, pages_end, total_words)
            patches.cv2_imshow(cv2.resize(img_bgr, (0, 0), fx=0.3, fy=0.3))
        conn.commit()

    clear_checkpoint()
    stats = {
        "timestamp": datetime.now().isoformat(),
        "pages": pages_end - pages_start + 1,
        "words": total_words
    }
    with open(STATS_JSON, 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False)
    logging.info(f"✅ اكتملت المعالجة في {time.time()-proc_start:.2f} ثانية. كلمات: {total_words}")